[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hoax12/fable-pytorch/blob/main/notebooks/14_final_boss.ipynb)

# ⚒️ Act III · Quest 14 — 👾 The Final Boss

*Everything you've forged, in one battle: data → model → evaluation → robustness → shipping.*

⬅️ [13_art_of_speed](13_art_of_speed.ipynb)  •  [15_beyond_the_forge_onnx](15_beyond_the_forge_onnx.ipynb) ➡️

---

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import math, random

torch.manual_seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch {torch.__version__} | device: {device}')
np.random.seed(0); random.seed(0)

# ══════════════════ ⚒️ FORGE GRADER — run me once ══════════════════
# Powers check() and xp_report(). Exercises give instant feedback + XP.
_XP = {"earned": 0, "done": set(), "checks": {}}

def _register(name, points, test, hint):
    _XP["checks"][name] = (points, test, hint)

def check(name, *answer):
    """Grade an exercise: check("ex1", your_answer). Repeatable until correct."""
    if name not in _XP["checks"]:
        print(f"unknown exercise: {name!r} — available: {list(_XP['checks'])}")
        return
    points, test, hint = _XP["checks"][name]
    try:
        ok = bool(test(*answer))
        err = None
    except Exception as e:
        ok, err = False, f"{type(e).__name__}: {e}"
    if ok:
        first = name not in _XP["done"]
        if first:
            _XP["done"].add(name)
            _XP["earned"] += points
        total = sum(p for p, _, _ in _XP["checks"].values())
        tag = f"+{points} XP" if first else "already earned"
        print(f"✅ {name} — correct! {tag}   ⚡ {_XP['earned']}/{total} XP")
    else:
        msg = f"❌ {name} — not yet."
        if err:
            msg += f" (your answer raised {err})"
        print(msg + f"\n   💡 hint: {hint}")

def xp_report():
    total = sum(p for p, _, _ in _XP["checks"].values())
    n = len(_XP["checks"])
    bar = "█" * int(10 * _XP["earned"] / max(total, 1)) or "░"
    print(f"⚡ XP: {_XP['earned']}/{total}   [{bar:<10}]   exercises: {len(_XP['done'])}/{n}")
    for name in _XP["checks"]:
        print(("  ✅ " if name in _XP["done"] else "  ⬜ ") + name)

## The battle

The boss has **5 health bars**. Each phase you clear knocks one out. Clear all five and you've
demonstrated end-to-end mastery: you can take a problem from raw data to a shipped artifact.

| Phase | Objective | Check |
|-------|-----------|-------|
| 1 · Supply lines | augmented train/test loaders | `phase1` |
| 2 · The weapon | model ≥ **94%** test accuracy | `phase2` |
| 3 · Battle damage | ≥ **75%** accuracy under heavy noise | `phase3` |
| 4 · Know your enemy | identify the most-confused glyph pair | `phase4` |
| 5 · Victory lap | TorchScript artifact that matches the model | `phase5` |

A **reference walkthrough** lives at the bottom — one honorable attempt before peeking.

In [ ]:
# --- The Glyph dataset: ✕ ◯ ┼ ╱  (self-contained, no torchvision needed) ----
GLYPHS = ["cross", "ring", "plus", "slash"]

def _draw_glyph(cls, size=20, rng=None):
    rng = rng or torch.Generator().manual_seed(0)
    img = torch.zeros(size, size)
    ys, xs = torch.meshgrid(torch.arange(size), torch.arange(size), indexing="ij")
    jx = int(torch.randint(-2, 3, (1,), generator=rng))   # random jitter
    jy = int(torch.randint(-2, 3, (1,), generator=rng))
    c = size // 2
    if cls == 0:    # cross ✕ : two diagonals
        img[((xs - ys).abs() + (jx - jy)).abs() <= 1] = 1.0
        img[((xs + ys - size + 1) + jx + jy).abs() <= 1] = 1.0
    elif cls == 1:  # ring ◯
        r2 = (xs - c - jx) ** 2 + (ys - c - jy) ** 2
        img[(r2 >= 25) & (r2 <= 49)] = 1.0
    elif cls == 2:  # plus ┼
        img[(ys - c - jy).abs() <= 1] = 1.0
        img[(xs - c - jx).abs() <= 1] = 1.0
    else:           # slash ╱ : one diagonal
        img[((xs + ys - size + 1) + jx + jy).abs() <= 1] = 1.0
    img = img + 0.08 * torch.randn(size, size, generator=rng)
    return img.clamp(0, 1)

def make_glyphs(n_per_class=300, size=20, seed=0):
    rng = torch.Generator().manual_seed(seed)
    X = torch.stack([_draw_glyph(c, size, rng) for c in range(4) for _ in range(n_per_class)])
    y = torch.arange(4).repeat_interleave(n_per_class)
    perm = torch.randperm(len(y), generator=rng)
    return X[perm].unsqueeze(1), y[perm]   # (N, 1, 20, 20), (N,)

In [ ]:
from torch.utils.data import Dataset, DataLoader

# The battlefield: fixed test set (never train on it!), and a harder noisy variant
X_test, y_test = make_glyphs(n_per_class=100, seed=999)
X_noisy = (X_test + 0.35 * torch.randn_like(X_test)).clamp(0, 1)

def test_acc(model, X=X_test, y=y_test):
    model.eval()
    with torch.no_grad():
        return (model(X.to(device)).argmax(1).cpu() == y).float().mean().item()

In [ ]:
# ── this notebook's exercises (run once) ───────────────────────────────
_BOSS = ["phase1", "phase2", "phase3", "phase4", "phase5"]
def boss_health():
    down = sum(1 for p in _BOSS if p in _XP["done"])
    hearts = "💜" * (5 - down) + "💥" * down
    print(f"👾 BOSS HEALTH: {hearts}   ({5 - down}/5 remaining)")
    if down == 5:
        print("🏆 THE BOSS FALLS. You are a PyTorch practitioner. Course complete!")

_register("phase1", 20,
    lambda tr, te: hasattr(tr, "batch_size") and hasattr(te, "batch_size")
                   and next(iter(tr))[0].shape[1:] == (1, 20, 20)
                   and len(tr.dataset) >= 800,
    "two DataLoaders over glyph data (>=800 train samples), batched, images shaped (1,20,20)")
_register("phase2", 30,
    lambda m: test_acc(m) >= 0.94,
    "train a CNN on your loaders until test accuracy >= 94% (augmentation + a few epochs does it)")
_register("phase3", 20,
    lambda m: test_acc(m, X_noisy, y_test) >= 0.75,
    "robustness: >= 75% on the noisy test set — train WITH noise augmentation")
_register("phase4", 15,
    lambda pair: set(map(str.lower, pair)) == {"cross", "slash"},
    "compute the confusion matrix on X_test; which two glyphs are confused most? submit like ('cross','slash')")
_register("phase5", 15,
    lambda ok: ok is True,
    "torch.jit.trace your model, save+reload, submit torch.allclose(model(x), reloaded(x)) for a test batch")

## ⚔️ Your battle — phases 1 through 5

In [ ]:
# Phase 1 — forge your data pipeline
# ...
# check("phase1", train_loader, test_loader); boss_health()

In [ ]:
# Phase 2 — forge and train your weapon
# ...
# check("phase2", model); boss_health()

In [ ]:
# Phase 3 — survive the noise storm
# check("phase3", model); boss_health()

# Phase 4 — study the confusion matrix
# check("phase4", ("?", "?")); boss_health()

# Phase 5 — ship it
# check("phase5", matches); boss_health()

---
## 🎓 Reference walkthrough (sensei's battle)

Every cell below runs — and defeats the boss. Compare it with your approach afterwards.

In [ ]:
# Phase 1 — supply lines: augmented training data
class BattleGlyphs(Dataset):
    def __init__(self, n=1600, noise_aug=0.30, seed=7):
        self.X, self.y = make_glyphs(n // 4, seed=seed)
        self.noise_aug = noise_aug
    def __len__(self):
        return len(self.y)
    def __getitem__(self, i):
        img, lbl = self.X[i], self.y[i]
        if random.random() < 0.5:
            img = img.flip(-1)
        img = img.roll(random.randint(-1, 1), dims=-1)
        img = (img + self.noise_aug * random.random() * torch.randn_like(img)).clamp(0, 1)
        return img, lbl

train_loader = DataLoader(BattleGlyphs(), batch_size=64, shuffle=True)
test_loader = DataLoader(torch.utils.data.TensorDataset(X_test, y_test), batch_size=256)
check("phase1", train_loader, test_loader)
boss_health()

In [ ]:
# Phase 2 — the weapon: a compact CNN with batchnorm
class BossSlayer(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.BatchNorm2d(16), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Flatten(), nn.Linear(32 * 5 * 5, 64), nn.ReLU(), nn.Dropout(0.2), nn.Linear(64, 4),
        )
    def forward(self, x):
        return self.net(x)

model = BossSlayer().to(device)
opt = torch.optim.Adam(model.parameters(), lr=1.5e-3)
for epoch in range(6):
    model.train()
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad(); F.cross_entropy(model(xb), yb).backward(); opt.step()
    print(f"epoch {epoch}: clean test {test_acc(model)*100:.1f}% | noisy test {test_acc(model, X_noisy, y_test)*100:.1f}%")
check("phase2", model)
boss_health()

In [ ]:
# Phase 3 — robustness came free with noise augmentation
check("phase3", model)
boss_health()

In [ ]:
# Phase 4 — know your enemy: the confusion matrix
model.eval()
with torch.no_grad():
    preds = model(X_test.to(device)).argmax(1).cpu()
cm = torch.zeros(4, 4, dtype=torch.int)
for t, p in zip(y_test, preds):
    cm[t, p] += 1

off = cm.clone(); off.fill_diagonal_(0)
i, j = divmod(int((off + off.T).argmax()), 4)
print("confusion matrix:\n", cm)
print(f"most-confused pair: {GLYPHS[i]} <-> {GLYPHS[j]}  (they share a diagonal stroke!)")
check("phase4", (GLYPHS[i], GLYPHS[j]))
boss_health()

In [ ]:
# Phase 5 — victory lap: ship a TorchScript artifact
import os
os.makedirs("checkpoints", exist_ok=True)
model.cpu().eval()
example = X_test[:4]
torch.jit.trace(model, example).save("checkpoints/boss_slayer.pt")
revived = torch.jit.load("checkpoints/boss_slayer.pt")
matches = torch.allclose(model(example), revived(example), atol=1e-5)
check("phase5", matches)
boss_health()
model.to(device)

xp_report()

---
# 🏆 Course complete

Look how far you've come:

- **Act I** — you *built* an autograd engine, then recognized it inside PyTorch.
- **Act II** — you gave networks eyes, memory, and attention. You built a GPT. You earned a
  debugging black belt.
- **Act III** — you sculpted hearts from noise, taught an agent to balance a pole, and shipped
  optimized artifacts.

### Where to next
- 🖥️ Re-run quests 07, 09, 11 on a **Colab GPU** with the scale knobs cranked up.
- 🕹️ Revisit **the Arcade** (`streamlit run arcade/Home.py`) — the demos will feel different now
  that you know what's underneath.
- 🌍 Pick a dataset you *care about* and repeat the Final Boss pipeline on it. That's the real
  final exam — and the beginning of your own work.

⚒️ *The forge is yours now.*